<a href="https://colab.research.google.com/github/ernestmwangombe/Auth-Login-protect/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ernestmwangombe/applied-ml-portfolio/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

A page should be considered for a refresh review if it gets a solid number of views, isn't brand new, and hasn't been updated for quite a while. Pages that are both highly visible and more out-of-date get higher priority for review.

**My rule**

The rule uses only three inputs: impressions in the last 90 days, days since the last update, and total content age. For each page, the rule calculates a score as:

score = impressions_90d × staleness_weight

where staleness_weight = days_since_last_update (capped at 180) / 180.

- **Eligibility:** at least 500 impressions in the last 90 days, at least 90 days since the last update, and content at least 90 days old. I chose these thresholds to focus on established, visible pages that might benefit most from a refresh, and to avoid noise from recent or little-seen content.
- **Reason code:** STALE_VISIBLE
- **Action:** refresh review

The label indicating whether a page is declining is used only in signal checks and evaluation, not as an input to the ranking.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, subprocess                      # os: files and folders; subprocess: runs the download command
import numpy as np                         # numpy: maths helpers
import pandas as pd                        # pandas: tables

DATA = "data/raw/content_refresh_anonymized.csv"                            # path to the starter data file
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"   # where to download it from
REPO_DIR = "flyrank-ml-internship-starter"                                  # folder name after download

if not os.path.exists(DATA):                                                # if this session doesn't have the data yet...
    if not os.path.isdir(REPO_DIR):                                         # ...and the folder isn't there either...
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)  # ...download it
    os.chdir(REPO_DIR)                                                      # move into the folder so paths work

df = pd.read_csv(DATA)                                                      # load the 30,000 pages into a table
df["declining"] = (df["trend_direction"].str.lower() == "down").astype(int) # OUTCOME ONLY: 1 if the page is declining
print(f"Pages: {len(df):,} | Base rate (share declining): {df['declining'].mean():.1%}")  # size and base rate

Pages: 30,000 | Base rate (share declining): 54.2%


In [ ]:
# Belief to test: "the longer since the last update, the more often a page declines"
bins = [-1, 30, 90, 180, np.inf]                                            # edges of the staleness groups, in days
names = ["0-30 days", "31-90 days", "91-180 days", "181+ days"]             # readable names for the groups
df["stale_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=names)  # put each page in a group

check1 = df.groupby("stale_bucket", observed=True).agg(      # for each group, calculate:
    n=("declining", "size"),                                  # n = how many pages are in the group
    declining_rate=("declining", "mean"),                     # share of those pages that are declining
).round(3)                                                    # round to 3 decimals
print(check1.to_string())                                     # show the bucket table
print("Pages with no staleness value:", df["days_since_last_update"].isna().sum())  # missing values

                  n  declining_rate
stale_bucket                       
0-30 days     20480           0.511
31-90 days      175           0.589
91-180 days    9171           0.611
181+ days       174           0.471
Pages with no staleness value: 0


**Signal check 1: staleness** (the signal behind FlyRank's refresh flags)

- **Belief:** Pages left untouched for longer are more likely to see declines.
- **What the table shows:** Declining rate is 51.1% (0–30 days since update, n=20,480), 58.9% (31–90 days, n=175), 61.1% (91–180 days, n=9,171), and 47.1% (181+ days, n=174).
- **Verdict: MIXED.** The declining rate rises with staleness up to 180 days, but then drops in the oldest group, which has few pages and may not be reliable.
- **Extra note:** Days since update takes only a handful of values: 20 and 104 days account for most pages (68%). The 104-day group covers 8,773 pages, making up almost all of the 91–180 group. This pattern is consistent with a site-wide update or sync across many pages.

In [ ]:
# Belief to test: "pages with more impressions are the ones worth ranking first (more at stake)"
bins = [-1, 0, 100, 500, 3000, np.inf]                                      # edges of the impression groups
names = ["0", "1-100", "101-500", "501-3,000", "3,000+"]                    # readable names for the groups
df["impr_bucket"] = pd.cut(df["impressions_90d"], bins=bins, labels=names)  # put each page in a group

check2 = df.groupby("impr_bucket", observed=True).agg(       # for each group, calculate:
    n=("declining", "size"),                                  # n = pages in the group
    declining_rate=("declining", "mean"),                     # share of them declining
).round(3)                                                    # round to 3 decimals
print(check2.to_string())                                     # show the bucket table

                n  declining_rate
impr_bucket                      
1-100        8006           0.389
101-500      5279           0.604
501-3,000    8432           0.621
3,000+       8283           0.570


**Signal check 2: volume**

- **Belief:** More visible pages are at greater risk of decline, so higher volume should correlate with a higher declining rate.
- **What the table shows:** Declining rate is 38.9% (1–100 impressions, n=8,006), 60.4% (101–500, n=5,279), 62.1% (501–3,000, n=8,432), and 57.0% (3,000+, n=8,283). Among the largest pages: 60.6% (3k–10k), 55.0% (10k–30k), 46.2% (30k+).
- **Verdict: MIXED.** The declining rate increases with volume at the low end but falls off among the biggest pages, suggesting that extremely high volume may be linked to greater stability.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

MIN_IMPR = 500       # gate 1: visible enough to matter (revisit after check 2, and write down why)
STALE_DAYS = 90      # gate 2: stale enough to review (revisit after check 1, and write down why)
MIN_AGE = 90         # gate 3: not brand new

eligible = ((df["impressions_90d"] >= MIN_IMPR)              # visible enough...
            & (df["days_since_last_update"] >= STALE_DAYS)   # ...and not updated for a while...
            & (df["content_age_days"] >= MIN_AGE))           # ...and not brand new (missing values count as False)

staleness_weight = df["days_since_last_update"].clip(upper=180) / 180      # 0 to 1: staler pages weigh more, capped at 180 days
df["score"] = np.where(eligible, df["impressions_90d"] * staleness_weight, 0.0)  # score = visibility x staleness (readable on purpose)
df["reason_code"] = np.where(eligible, "STALE_VISIBLE", "")                # the ONE reason code
df["action"] = np.where(eligible, "refresh review", "none")                # the action label

queue = df[eligible].sort_values("score", ascending=False).copy()          # keep eligible pages, highest score first
queue["rank"] = np.arange(1, len(queue) + 1)                               # rank 1, 2, 3, ...

cols = ["rank", "content_id", "client_id", "score", "reason_code", "action",
        "impressions_90d", "days_since_last_update", "content_age_days", "avg_position", "ctr"]  # columns to save
cols = [c for c in cols if c in queue.columns]                             # keep only columns that exist
os.makedirs("work/outputs", exist_ok=True)                                 # make the output folder if needed
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)  # write the ranked queue as a CSV
print(f"Eligible pages in the queue: {len(queue):,} of {len(df):,}")       # coverage: how many pages the rule ranks

Eligible pages in the queue: 6,575 of 30,000


The rule ranks 6,575 of 30,000 pages (21.9%) and writes the ranked queue to work/outputs/baseline_action_score.csv.

In [ ]:
import json                                                                # json: saves a small results file
base = df["declining"].mean()                                              # base rate: what random picking gives
metrics = {"base_rate": round(float(base), 3), "eligible_pages": int(len(queue))}   # start the results record

for k in (10, 20, 50):                                                     # check the top 10, 20 and 50
    if len(queue) >= k:                                                    # only if the queue is long enough
        p = queue["declining"].head(k).mean()                              # share of the top k that really are declining
        metrics[f"precision_at_{k}"] = round(float(p), 3)                  # remember it
        print(f"Precision@{k}: {p:.3f}   (base rate {base:.3f})")          # print it next to the base rate

json.dump(metrics, open("work/outputs/baseline_metrics.json", "w"), indent=2)  # save the results (optional to commit)

Precision@10: 0.600   (base rate 0.542)
Precision@20: 0.450   (base rate 0.542)
Precision@50: 0.440   (base rate 0.542)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review = ["rank", "content_type", "impressions_90d", "days_since_last_update",
          "content_age_days", "word_count", "avg_position", "ctr", "score"]   # facts a reviewer needs
review = [c for c in review if c in queue.columns]                           # keep only columns that exist
print(queue[review].head(20).round(2).to_string(index=False))   # show the top 20 as plain text

 rank    content_type  impressions_90d  days_since_last_update  content_age_days  word_count  avg_position  ctr     score
    1 keyword article           517715                     104               537         NaN           4.2 0.14 299124.22
    2 keyword article           443434                     104               299      7676.0          27.9 0.21 256206.31
    3 keyword article           347399                     104               362         NaN           4.2 0.53 200719.42
    4 keyword article           309910                     104               126      2761.0           5.6 0.16 179059.11
    5 keyword article           309192                     104               445         NaN           2.0 0.87 178644.27
    6 keyword article           295097                     104               144         NaN           7.3 0.05 170500.49
    7 keyword article           286608                     104               153      6901.0          26.2 0.06 165595.73
    8 keyword article   

1. 517,715 impr · 104 d · age 537 · pos 4.2 · CTR 0.14 · words missing → refresh review | Why: High visibility and long time since update | Wrong if: The 104-day staleness is a result of a mass update, not true neglect
2. 443,434 impr · 104 d · age 299 · pos 27.9 · CTR 0.21 · 7,676 words → refresh review | Why: Meets all rule thresholds; very visible and stale | Wrong if: Position near 28 plus huge impressions could mean a tracking or reporting error
3. 347,399 impr · 104 d · age 362 · pos 4.2 · CTR 0.53 · words missing → refresh review | Why: Old, well-seen, and not updated in months | Wrong if: Missing word count hides whether it is thin content
4. 309,910 impr · 104 d · age 126 · pos 5.6 · CTR 0.16 · 2,761 words → refresh review | Why: All criteria met: visible and stale | Wrong if: Same 104-day date as many other pages, possibly a bulk update
5. 309,192 impr · 104 d · age 445 · pos 2.0 · CTR 0.87 · words missing → refresh review | Why: Very old and frequently viewed | Wrong if: Missing word count and a bulk 104-day update
6. 295,097 impr · 104 d · age 144 · pos 7.3 · CTR 0.05 · words missing → refresh review | Why: High impressions and stale | Wrong if: Missing word count and a bulk update date
7. 286,608 impr · 104 d · age 153 · pos 26.2 · CTR 0.06 · 6,901 words → refresh review | Why: Lengthy, visible, and not updated in a while | Wrong if: A "sibling" page with identical stats might be in the list
8. 233,561 impr · 104 d · age 153 · pos 26.2 · CTR 0.06 · 4,610 words → refresh review | Why: Passes all thresholds | Wrong if: Possibly a duplicate or sibling page
9. 211,366 impr · 104 d · age 126 · pos 5.1 · CTR 0.41 · 2,874 words → refresh review | Why: Substantial, well-trafficked, and stale | Wrong if: Part of the 104-day cohort, so possibly a bulk update
10. 208,678 impr · 104 d · age 362 · pos 9.7 · CTR 0.00 · words missing → refresh review | Why: Old, high impressions | Wrong if: A CTR of 0.00 suggests a possible tracking problem
11. 205,915 impr · 104 d · age 216 · pos 27.9 · CTR 0.14 · 5,591 words → refresh review | Why: Big, long, and not updated | Wrong if: Position near 28 with high impressions looks odd
12. 201,584 impr · 104 d · age 313 · pos 5.8 · CTR 0.24 · words missing → refresh review | Why: Old and visible | Wrong if: Missing word count and a bulk 104-day update
13. 201,111 impr · 104 d · age 257 · pos 5.7 · CTR 0.11 · words missing → refresh review | Why: High volume, high age | Wrong if: Missing word count and a bulk update
14. 192,205 impr · 104 d · age 211 · pos 12.5 · CTR 0.24 · 5,127 words → refresh review | Why: Size and staleness meet thresholds | Wrong if: Same 104-day date as many pages, so possibly a bulk update
15. 190,623 impr · 104 d · age 124 · pos 4.3 · CTR 0.24 · 2,824 words → refresh review | Why: Not updated in a while, good traffic | Wrong if: Bulk update date
16. 187,893 impr · 104 d · age 313 · pos 4.0 · CTR 0.45 · words missing → refresh review | Why: Old and well-trafficked | Wrong if: Missing word count and a bulk update date
17. 181,574 impr · 104 d · age 144 · pos 6.2 · CTR 0.16 · words missing → refresh review | Why: Passes all cut-offs | Wrong if: Bulk update date and missing word count
18. 181,514 impr · 104 d · age 223 · pos 25.8 · CTR 0.10 · 5,876 words → refresh review | Why: High word count, visible, stale | Wrong if: Pattern of a shared 104-day update
19. 179,002 impr · 104 d · age 299 · pos 22.1 · CTR 0.08 · 6,974 words → refresh review | Why: All rule criteria met | Wrong if: Bulk 104-day update date
20. 176,296 impr · 104 d · age 236 · pos 4.3 · CTR 0.23 · words missing → refresh review | Why: Old, visible, not updated | Wrong if: Missing word count and a bulk update

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# How many distinct "days since last update" values are there, and which are most common?
print(df["days_since_last_update"].value_counts().head(8).to_string())   # the 8 most common values and their counts

# Does the declining rate differ among the very biggest pages? (finer volume groups)
big = df[df["impressions_90d"] >= 3000].copy()                           # keep only pages with 3,000+ impressions
big["big_bucket"] = pd.cut(big["impressions_90d"], bins=[2999, 10000, 30000, np.inf],
                           labels=["3k-10k", "10k-30k", "30k+"])         # three finer groups
print(big.groupby("big_bucket", observed=True)["declining"].agg(["size", "mean"]).round(3).to_string())  # n and declining rate

days_since_last_update
20     11573
104     8773
22      3564
8       1929
13       515
14       505
28       466
25       441
            size   mean
big_bucket             
3k-10k      4681  0.606
10k-30k     2524  0.550
30k+        1078  0.462


**Weak picks**

- **Weak pick 1:** The top 20 are all extremely large pages (176,000+ impressions), yet pages in the 30k+ group have the lowest declining rate (46.2%), below the base rate of 54.2%.
- **Weak pick 2:** Rank 10 has over 200,000 impressions but a CTR of 0.00, which may be a tracking error.
- **What it means for the baseline:** Precision@10 is 0.600, @20 is 0.450 and @50 is 0.440, against a base rate of 0.542. Overall that is no better than random picking, and below it at 20 and 50. This is the baseline the Week 5 model has to beat.

**Leakage check**

- **Inputs used:** impressions_90d, days_since_last_update, content_age_days. Not used: trend_direction, trend_pct, any product flags.
- **Caveat:** The starter data is a single snapshot, and impressions_90d covers the same 90 days as the label (last 30 vs previous 30), so this is an association check, not a forecast.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.